In [1]:
import polars as pl
import gc
import glob

In [3]:
target_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\*.csv"
file_paths = glob.glob(target_path)
print(file_paths)
# file_name = file_paths[0].split('/')[-1]
# print(file_name)

['C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset1.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset10.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset11.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset12.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset13.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset14.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset15.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset16.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset17.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset18.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset19.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset2

In [26]:
target_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\*.csv"
file_paths = glob.glob(target_path)

print(f"Found {len(file_paths)} datasets. Running surgical audit...\n")
total_malicious = 0
# Loop through each file, but ONLY scan the final messy column
for file in sorted(file_paths):
    file_name = file.split('\\')[-1]
    
    try:
        # Lazy scan, selecting only the column we need to parse
        lf = pl.scan_csv(
            file,
            ignore_errors=True,
            null_values=["-", "(empty)"]
        ).select([
            pl.col("tunnel_parents   label   detailed-label").str.split("   ").list.get(1).alias("label")
        ])
        
        # Trigger the execution for this specific file
        counts = lf.group_by("label").count().collect()
        
        # Extract numbers safely
        labels_present = counts["label"].to_list()
        benign_count = counts.filter(pl.col("label") == "Benign")["count"][0] if "Benign" in labels_present else 0
        malicious_count = counts.filter(pl.col("label") != "Benign")["count"].sum()
        
        print(f"{file_name:<15} -> Benign: {benign_count:<10} | Malicious: {malicious_count:<10}")
        
        total_malicious += malicious_count

    except Exception as e:
        print(f"{file_name:<15} -> ERROR reading file: {e}")

print(f"\nTotal Malicious Flows Across All Files: {total_malicious}")

Found 23 datasets. Running surgical audit...

dataset1.csv    -> Benign: 0          | Malicious: 452       


C:\Users\babai\AppData\Local\Temp\ipykernel_28064\389290745.py:21: DeprecationWarning: `count` was renamed; use `len` instead
  counts = lf.group_by("label").count().collect()


dataset10.csv   -> Benign: 3734       | Malicious: 3390604   
dataset11.csv   -> Benign: 211        | Malicious: 26        
dataset12.csv   -> Benign: 20574934   | Malicious: 46746875  
dataset13.csv   -> Benign: 4420       | Malicious: 6         
dataset14.csv   -> Benign: 7337       | Malicious: 73561644  
dataset15.csv   -> Benign: 2663       | Malicious: 13642435  
dataset16.csv   -> Benign: 8262389    | Malicious: 2185398   
dataset17.csv   -> Benign: 1923       | Malicious: 21222     
dataset18.csv   -> Benign: 1380791    | Malicious: 53073800  
dataset19.csv   -> Benign: 4536       | Malicious: 151567    
dataset2.csv    -> Benign: 0          | Malicious: 1374      
dataset20.csv   -> Benign: 3272       | Malicious: 14        
dataset21.csv   -> Benign: 3193       | Malicious: 16        
dataset22.csv   -> Benign: 31438      | Malicious: 54628417  
dataset23.csv   -> Benign: 469275     | Malicious: 539473    
dataset3.csv    -> Benign: 0          | Malicious: 130       
dataset4

In [4]:
import os

#borrowed schema from https://www.kaggle.com/code/harishragavender0209/iot23-peoject
IOT23_SCHEMA = {
    'ts': pl.Float64,              # Timestamp needs precision
    'uid': pl.String,              # Connection unique ID
    'id.orig_h': pl.String,        # Source IP
    'id.orig_p': pl.UInt32,        # Source Port (Unsigned Int, ports can't be negative)
    'id.resp_h': pl.String,        # Destination IP
    'id.resp_p': pl.UInt32,        # Destination Port
    'proto': pl.Categorical,       # Protocol (tcp, udp, icmp) - Categorical saves massive RAM
    'service': pl.String,          # http, dns, ssh, etc. 
    'duration': pl.Float32,        # Downcast to Float32
    'orig_bytes': pl.String,       # Read as string first because of '-' nulls
    'resp_bytes': pl.String,       
    'conn_state': pl.Categorical,  # S0, SF, REJ, etc.
    'local_orig': pl.String,       
    'local_resp': pl.String,       
    'missed_bytes': pl.Float32,    
    'history': pl.String,          
    'orig_pkts': pl.Float32,       
    'orig_ip_bytes': pl.Float32,   
    'resp_pkts': pl.Float32,       
    'resp_ip_bytes': pl.Float32,   
    # The messy column as a single string first
    'tunnel_parents   label   detailed-label': pl.String 
}

def parse_and_clean_schema(file_path, file_name, chunk_path, typ):
    lf = pl.scan_csv(
        file_path,
        schema=IOT23_SCHEMA,
        ignore_errors=True,
        null_values=["-", "(empty)"]
    )

    lf_cleaned = lf.with_columns([
        pl.col("orig_bytes").cast(pl.Float32),
        pl.col("resp_bytes").cast(pl.Float32),
        pl.col("tunnel_parents   label   detailed-label").str.split(by="   ").alias("split_target")
        #splitting the messy column into three separate columns
    ]).with_columns([pl.col("split_target").list.get(0).alias("tunnel_parents"),
    pl.col("split_target").list.get(1).alias("label").cast(pl.Categorical),
    pl.col("split_target").list.get(2).alias("detailed-label").cast(pl.Categorical)]
    #categorical for memory efficiency
    ).drop(["tunnel_parents   label   detailed-label", "split_target"])

    if typ == "benign":
        lf_benign = lf_cleaned.filter(pl.col("label").is_in(["Benign", "benign"]))
    elif typ == "malicious":
        lf_benign = lf_cleaned.filter(~pl.col("label").is_in(["Benign", "benign"]))
    df_benign = lf_benign.collect()

    if df_benign.height > 0:
        df_benign.write_parquet(chunk_path)
        print(f"Saved {df_benign.height} {typ} rows from {file_name}.")
    else:
        print(f"Skipped {file_name} (0 Benign rows).")

    del df_benign
    gc.collect()


In [28]:
temp_dir = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_chunks"
os.makedirs(temp_dir, exist_ok=True)

for file in sorted(file_paths):
    file_name = os.path.basename(file)
    chunk_path = os.path.join(temp_dir, f"benign_{file_name.replace('.csv', '.parquet')}")

    parse_and_clean_schema(file, file_name, chunk_path, typ="benign")

final_benign_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_master.parquet"
pl.scan_parquet(f"{temp_dir}/*.parquet").sink_parquet(final_benign_path)

print("successfully merged all benign chunks into a master parquet file.")

Saved 452 benign rows from dataset1.csv.
Saved 3734 benign rows from dataset10.csv.
Saved 211 benign rows from dataset11.csv.
Saved 20574934 benign rows from dataset12.csv.
Saved 4420 benign rows from dataset13.csv.
Saved 7337 benign rows from dataset14.csv.
Saved 2663 benign rows from dataset15.csv.
Saved 8262389 benign rows from dataset16.csv.
Saved 1923 benign rows from dataset17.csv.
Saved 1380791 benign rows from dataset18.csv.
Saved 4536 benign rows from dataset19.csv.
Saved 1374 benign rows from dataset2.csv.
Saved 3272 benign rows from dataset20.csv.
Saved 3193 benign rows from dataset21.csv.
Saved 31438 benign rows from dataset22.csv.
Saved 469275 benign rows from dataset23.csv.
Saved 130 benign rows from dataset3.csv.
Saved 22548 benign rows from dataset4.csv.
Saved 2181 benign rows from dataset5.csv.
Saved 75955 benign rows from dataset6.csv.
Saved 2476 benign rows from dataset7.csv.
Saved 1794 benign rows from dataset8.csv.
Saved 3665 benign rows from dataset9.csv.
successf

In [6]:
final_benign_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_master.parquet" #redecalared for block running
total_benign = pl.scan_parquet(final_benign_path).select(pl.len()).collect().item()
target_malicious = total_benign * 3
print(f"Total Benign Flows: {total_benign}")
print(f"Target Malicious Flows (1:3): {target_malicious}")

total_malicious = 294451211 
sample_fraction = target_malicious / total_malicious
hash_threshold = int(sample_fraction * 100000)

temp_dir = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_chunks"
os.makedirs(temp_dir, exist_ok=True)

for file in sorted(file_paths):
    file_name = os.path.basename(file)
    chunk_path = os.path.join(temp_dir, f"malicious_{file_name.replace('.csv', '.parquet')}")

    parse_and_clean_schema(file, file_name, chunk_path, typ="malicious")

final_malicious_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_master.parquet"
pl.scan_parquet(f"{temp_dir}/*.parquet").sink_parquet(final_malicious_path)

print("successfully merged all malicious chunks into a master parquet file.")

Total Benign Flows: 30860691
Target Malicious Flows (1:3): 92582073
Skipped dataset1.csv (0 Benign rows).
Saved 3390604 malicious rows from dataset10.csv.
Saved 26 malicious rows from dataset11.csv.
Saved 46746875 malicious rows from dataset12.csv.
Saved 6 malicious rows from dataset13.csv.
Saved 73561644 malicious rows from dataset14.csv.
Saved 13642435 malicious rows from dataset15.csv.
Saved 2185398 malicious rows from dataset16.csv.
Saved 21222 malicious rows from dataset17.csv.
Saved 53073800 malicious rows from dataset18.csv.
Saved 151567 malicious rows from dataset19.csv.
Skipped dataset2.csv (0 Benign rows).
Saved 14 malicious rows from dataset20.csv.
Saved 16 malicious rows from dataset21.csv.
Saved 54628417 malicious rows from dataset22.csv.
Saved 539473 malicious rows from dataset23.csv.
Skipped dataset3.csv (0 Benign rows).
Saved 6355745 malicious rows from dataset4.csv.
Saved 8222 malicious rows from dataset5.csv.
Saved 11378759 malicious rows from dataset6.csv.
Saved 3578

In [2]:
def clean_and_impute(input_path, output_path):
    
    lf = pl.scan_parquet(input_path)
    existing_cols = lf.collect_schema().names()

    cols_to_drop = ["uid", "local_orig", "local_resp"]
    actual_drops = [col for col in cols_to_drop if col in existing_cols]
    lf = lf.drop(actual_drops)

    num_cols = [
        "duration", "orig_bytes", "resp_bytes", "missed_bytes", 
        "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"
    ]
    cat_cols = ["service", "history"]

    num_imputes = [pl.col(c).fill_null(0.0) for c in num_cols if c in existing_cols]
    cat_imputes = [pl.col(c).fill_null("unknown") for c in cat_cols if c in existing_cols]

    lf_cleaned = lf.with_columns(num_imputes + cat_imputes)

    print(f"Streaming cleaned data to {output_path}...")
    lf_cleaned.sink_parquet(output_path)
    
    print(f"SUCCESS: Cleaned dataset saved -> {output_path}\n")

In [3]:
clean_and_impute("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_master.parquet", "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_cleaned.parquet")
gc.collect()

clean_and_impute("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_master.parquet", "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_cleaned.parquet")
gc.collect()

Streaming cleaned data to C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\benign_cleaned.parquet...
SUCCESS: Cleaned dataset saved -> C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\benign_cleaned.parquet

Streaming cleaned data to C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\malicious_cleaned.parquet...
SUCCESS: Cleaned dataset saved -> C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\malicious_cleaned.parquet



0

In [2]:
def fuse_and_structure():
    
    lf_benign = pl.scan_parquet("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_cleaned.parquet")
    lf_malicious = pl.scan_parquet("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_cleaned.parquet")
    
    lf_combined = pl.concat([lf_benign, lf_malicious])
    
    selected_columns = [
        "ts",             # MUST keep for sorting and temporal splits
        "id.orig_p", "id.resp_p", "proto", "duration", 
        "orig_bytes", "resp_bytes", "conn_state", 
        "history", "orig_pkts", "resp_pkts",
        "label", "detailed-label" # Targets
    ]
    
    print("Applying chronological sort (this is computationally heavy)...")
    lf_final = (
        lf_combined
        .select(selected_columns)
        .sort("ts") 
    )
    
    output_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\iot23_master_chronological.parquet"
    lf_final.sink_parquet(output_path)
    
    print(f"SUCCESS: Master chronological dataset saved to {output_path}")

In [3]:
fuse_and_structure()
gc.collect()

Applying chronological sort (this is computationally heavy)...
SUCCESS: Master chronological dataset saved to C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\iot23_master_chronological.parquet


0